# LLM Slot Extraction - Local Setup (macOS M1 Pro)

**Purpose**: Run Qwen3-4B-Instruct-2507 locally with native llama.cpp + Metal acceleration

**Model**: `Qwen3-4B-Instruct-2507-Q4_K_M.gguf` (~2.6GB, Q4_K_M quantization)

**System Prompt Version**: v2.0 - 10 Tasks with Full Bahasa Indonesia Support

**Requirements**:
- macOS with Apple Silicon (M1/M2/M3)
- `brew install llama.cpp`
- ~3GB free disk space for model

---

## Architecture

```
┌─────────────────┐    HTTP/JSON     ┌──────────────────┐
│  Python Client  │ ───────────────> │  llama-server    │
│  (this notebook)│                  │  (native, Metal) │
└─────────────────┘                  └──────────────────┘
                                              │
                                              ▼
                                     ┌──────────────────┐
                                     │ Qwen3-4B-2507    │
                                     │ (Q4_K_M, ~2.6GB) │
                                     └──────────────────┘
```

## 10 Supported Tasks

1. `check_order` - Status pesanan, tracking
2. `product_info` - Detail produk, spesifikasi
3. `product_list` - Daftar produk umum
4. `search_product` - Cari via keyword
5. `product_by_category` - Filter kategori
6. `product_by_attribute` - Filter brand/warna/ukuran
7. `ask_price` - Harga produk
8. `check_stock` - Ketersediaan stok
9. `ask_payment` - Metode pembayaran
10. `out_of_scope` - Di luar cakupan

## Step 1: Verify llama.cpp Installation

In [10]:
import subprocess
import shutil

# Check if llama-server is installed
llama_server = shutil.which("llama-server")
llama_cli = shutil.which("llama-cli")

if llama_server:
    print(f"llama-server found: {llama_server}")
else:
    print("ERROR: llama-server not found!")
    print("Install with: brew install llama.cpp")

if llama_cli:
    result = subprocess.run(["llama-cli", "--version"], capture_output=True, text=True)
    print(f"Version: {result.stdout.strip() or result.stderr.strip()}")

llama-server found: /opt/homebrew/bin/llama-server
Version: ggml_metal_device_init: tensor API disabled for pre-M5 and pre-A19 devices
ggml_metal_library_init: using embedded metal library
ggml_metal_library_init: loaded in 14.791 sec
ggml_metal_device_init: GPU name:   Apple M1 Pro
ggml_metal_device_init: GPU family: MTLGPUFamilyApple7  (1007)
ggml_metal_device_init: GPU family: MTLGPUFamilyCommon3 (3003)
ggml_metal_device_init: GPU family: MTLGPUFamilyMetal3  (5001)
ggml_metal_device_init: simdgroup reduction   = true
ggml_metal_device_init: simdgroup matrix mul. = true
ggml_metal_device_init: has unified memory    = true
ggml_metal_device_init: has bfloat            = true
ggml_metal_device_init: has tensor            = false
ggml_metal_device_init: use residency sets    = true
ggml_metal_device_init: use shared buffers    = true
ggml_metal_device_init: recommendedMaxWorkingSetSize  = 11453.25 MB
version: 7270 (87a2084c4)
built with AppleClang 16.0.0.16000026 for Darwin arm64


## Step 2: Download Model (if needed)

In [13]:
import os
from pathlib import Path

# Get project root
NOTEBOOK_DIR = Path(os.getcwd())
PROJECT_ROOT = NOTEBOOK_DIR.parent
MODEL_DIR = PROJECT_ROOT / "models" / "llm"
MODEL_FILE = MODEL_DIR / "Qwen3-4B-Instruct-2507-Q4_K_M.gguf"

MODEL_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Model file: {MODEL_FILE}")
print(f"Model exists: {MODEL_FILE.exists()}")

if MODEL_FILE.exists():
    print(f"Size: {MODEL_FILE.stat().st_size / 1e9:.2f} GB")

Project root: /Users/firas/Developer/skripsi-balor
Model file: /Users/firas/Developer/skripsi-balor/models/llm/Qwen3-4B-Instruct-2507-Q4_K_M.gguf
Model exists: True
Size: 2.50 GB


In [14]:
# Download model if not exists
if not MODEL_FILE.exists():
    print("Downloading Qwen3-4B-Instruct-2507 Q4_K_M (~2.6 GB)...")
    print("This may take 5-15 minutes depending on connection speed.")
    
    from huggingface_hub import hf_hub_download
    
    model_path = hf_hub_download(
        repo_id="Qwen/Qwen3-4B-Instruct-2507-GGUF",
        filename="Qwen3-4B-Instruct-2507-Q4_K_M.gguf",
        local_dir=str(MODEL_DIR)
    )
    
    print(f"\nDownload complete: {model_path}")
else:
    print(f"Model already downloaded: {MODEL_FILE}")

Model already downloaded: /Users/firas/Developer/skripsi-balor/models/llm/Qwen3-4B-Instruct-2507-Q4_K_M.gguf


## Step 3: Start llama-server

**Run this in a separate terminal:**

```bash
# Option 1: Use the helper script
./scripts/start_llm_server.sh

# Option 2: Run directly
llama-server -m models/Qwen3-4B-Instruct-2507-Q4_K_M.gguf -c 4096 -ngl 99 --port 8080
```

Wait until you see: `llama server listening at http://0.0.0.0:8080`

In [13]:
# Check if server is running
import requests

SERVER_URL = "http://localhost:8080"

try:
    response = requests.get(f"{SERVER_URL}/health", timeout=5)
    if response.status_code == 200:
        print(f"Server is running at {SERVER_URL}")
        print(f"Health: {response.json()}")
    else:
        print(f"Server returned status: {response.status_code}")
except requests.exceptions.ConnectionError:
    print("Server is NOT running!")
    print("")
    print("Start the server in a separate terminal:")
    print(f"  ./scripts/start_llm_server.sh")
    print("")
    print("Or manually:")
    print(f"  llama-server -m {MODEL_FILE} -c 4096 -ngl 99 --port 8080")

Server is running at http://localhost:8080
Health: {'status': 'ok'}


## Step 4: Define Slot Extraction Client

This cell defines the `SlotExtractor` class with:
- **10 valid tasks** for classification
- **System prompt v2.0** with full Bahasa Indonesia support
- **New entity structure** with nested `attributes` (brand, color, size)
- **Retry logic** for handling invalid responses
- **Multi-intent detection** support

In [15]:
import json
import re
import requests
from typing import Dict, Any, Optional

# Valid task types for validation (10 tasks)
VALID_TASKS = {
    'check_order',
    'product_info',
    'product_list',
    'search_product',
    'product_by_category',
    'product_by_attribute',
    'ask_price',
    'check_stock',
    'ask_payment',
    'out_of_scope'
}

# Required fields in response
REQUIRED_FIELDS = {'task', 'entities', 'confidence', 'needs_clarification'}

# System prompt for slot extraction with few-shot examples (10 tasks, full Bahasa Indonesia)
SYSTEM_PROMPT = """
Kamu adalah AI e-commerce chatbot yang bertugas untuk **memahami, mengklasifikasikan, dan mengekstrak intent serta entity dari pertanyaan pengguna** dalam Bahasa Indonesia (formal maupun informal).

- TUGAS UTAMA kamu adalah **intent detection & entity extraction**, BUKAN menjawab pertanyaan.
- Output kamu akan diproses oleh backend (database & business logic).

---

**OUTPUT FORMAT (WAJIB JSON SAJA, TANPA TEKS TAMBAHAN):**

{
  "task": "check_order" |
          "product_info" |
          "product_list" |
          "search_product" |
          "product_by_category" |
          "product_by_attribute" |
          "ask_price" |
          "check_stock" |
          "ask_payment" |
          "out_of_scope",

  "entities": {
    "product_name": "string or null",
    "category": "string or null",
    "attributes": {
      "brand": "string or null",
      "color": "string or null",
      "size": "string or null"
    },
    "order_id": "string or null",
    "quantity": "number or null"
  },

  "multi_intent": ["list of secondary intents if any"],

  "confidence": 0.0 - 1.0,

  "needs_clarification": true | false,

  "clarification_question": "string or null"
}

---

**TASK DEFINITIONS (TOTAL: 10 — SEMUA WAJIB DIPATUHI):**

1. check_order
   - Status pesanan, detail order, status pembayaran, tracking pengiriman
   - order_id WAJIB
   - Jika order_id tidak ada → needs_clarification = true

2. product_info
   - Informasi detail produk (deskripsi, bahan, spesifikasi)

3. product_list
   - Menanyakan produk apa saja yang tersedia secara umum (eksploratif)

4. search_product
   - Mencari produk/kategori via keyword langsung

5. product_by_category
   - Produk berdasarkan kategori tertentu

6. product_by_attribute
   - Produk berdasarkan atribut (brand, warna, ukuran, dll)

7. ask_price
   - Menanyakan harga produk spesifik

8. check_stock
   - Menanyakan ketersediaan stok produk

9. ask_payment
   - Menanyakan metode atau cara pembayaran

10. out_of_scope
    - Di luar cakupan sistem (refund, komplain, jam buka toko, dll)
    - Akan difallback ke admin / WhatsApp

---

NORMALISASI BAHASA INFORMAL:
- aku/gue/gw/lo = saya
- ga/gak/nggak = tidak
- gimana/gmn = bagaimana
- brp/hrg = berapa/harga
- psen/psn = pesanan
- kak = kakak
- dan Bahasa Indonesia informal yang lainnya, mencakup juga typo dan bahasa gaul

---

**CRITICAL RULES:**

1. WAJIB output JSON lengkap
2. DILARANG output teks lain
3. Semua field HARUS ADA (isi null jika tidak dikenal)
4. Jangan mengarang entity yang tidak disebut user
5. Intent ambigu → out_of_scope + confidence 0.5
6. Data penting hilang → needs_clarification = true
7. clarification_question harus singkat & jelas
8. Jangan menjawab sebagai customer service
9. confidence harus mencerminkan kejelasan intent

---

FEW-SHOT EXAMPLES (MENCAKUP SEMUA 10 TASK):

────────────────────────
TASK: ask_payment
User: "bisa bayar pake gopay ga?"
{"task": "ask_payment", "entities": {"product_name": null, "category": null, "attributes": {"brand": null, "color": null, "size": null}, "order_id": null, "quantity": null}, "multi_intent": [], "confidence": 0.9, "needs_clarification": false, "clarification_question": null}

────────────────────────
TASK: check_order
User: "pesanan 12345 udah sampai mana?"
{"task": "check_order", "entities": {"product_name": null, "category": null, "attributes": {"brand": null, "color": null, "size": null}, "order_id": "12345", "quantity": null}, "multi_intent": [], "confidence": 0.95, "needs_clarification": false, "clarification_question": null}

────────────────────────
TASK: product_info
User: "bahan kaos ini apa?"
{"task": "product_info", "entities": {"product_name": "kaos", "category": null, "attributes": {"brand": null, "color": null, "size": null}, "order_id": null, "quantity": null}, "multi_intent": [], "confidence": 0.9, "needs_clarification": false, "clarification_question": null}

────────────────────────
TASK: product_list
User: "produk ada apa aja?"
{"task": "product_list", "entities": {"product_name": null, "category": null, "attributes": {"brand": null, "color": null, "size": null}, "order_id": null, "quantity": null}, "multi_intent": [], "confidence": 0.85, "needs_clarification": false, "clarification_question": null}

────────────────────────
TASK: search_product
User: "cari sepatu lari dong"
{"task": "search_product", "entities": {"product_name": "sepatu lari", "category": null, "attributes": {"brand": null, "color": null, "size": null}, "order_id": null, "quantity": null}, "multi_intent": [], "confidence": 0.9, "needs_clarification": false, "clarification_question": null}

────────────────────────
TASK: product_by_category
User: "produk fashion ada apa aja?"
{"task": "product_by_category", "entities": {"product_name": null, "category": "fashion", "attributes": {"brand": null, "color": null, "size": null}, "order_id": null, "quantity": null}, "multi_intent": [], "confidence": 0.9, "needs_clarification": false, "clarification_question": null}

────────────────────────
TASK: product_by_attribute
User: "ada baju nike warna merah ukuran L?"
{"task": "product_by_attribute", "entities": {"product_name": "baju", "category": null, "attributes": {"brand": "nike", "color": "merah", "size": "L"}, "order_id": null, "quantity": null}, "multi_intent": [], "confidence": 0.95, "needs_clarification": false, "clarification_question": null}

────────────────────────
TASK: ask_price
User: "harga laptop asus berapa?"
{"task": "ask_price", "entities": {"product_name": "laptop", "category": null, "attributes": {"brand": "asus", "color": null, "size": null}, "order_id": null, "quantity": null}, "multi_intent": [], "confidence": 0.95, "needs_clarification": false, "clarification_question": null}

────────────────────────
TASK: check_stock
User: "stok hp xiaomi masih ada?"
{"task": "check_stock", "entities": {"product_name": "hp", "category": null, "attributes": {"brand": "xiaomi", "color": null, "size": null}, "order_id": null, "quantity": null}, "multi_intent": [], "confidence": 0.9, "needs_clarification": false, "clarification_question": null}

────────────────────────
TASK: out_of_scope
User: "mau refund barang rusak"
{"task": "out_of_scope", "entities": {"product_name": null, "category": null, "attributes": {"brand": null, "color": null, "size": null}, "order_id": null, "quantity": null}, "multi_intent": [], "confidence": 0.9, "needs_clarification": false, "clarification_question": null}

====================================================================

Sekarang proses input berikut dan outputkan JSON sesuai aturan di atas.
"""


class SlotExtractor:
    """Client for llama-server slot extraction with retry logic."""
    
    def __init__(self, server_url: str = "http://localhost:8080"):
        self.server_url = server_url
        self.api_url = f"{server_url}/v1/chat/completions"
    
    def _validate_response(self, result: dict) -> bool:
        """Validate that response has all required fields and valid task."""
        if not result or not isinstance(result, dict):
            return False
        if not result.get('task'):
            return False
        if result.get('task') not in VALID_TASKS:
            return False
        if not all(field in result for field in REQUIRED_FIELDS):
            return False
        return True
    
    def _do_extract(self, query: str, temperature: float) -> tuple:
        """Perform single extraction attempt."""
        payload = {
            "model": "qwen3-4b-instruct-2507",
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": query}
            ],
            "max_tokens": 512,
            "temperature": temperature,
            "top_p": 0.9,
            "response_format": {"type": "json_object"}
        }
        
        try:
            response = requests.post(self.api_url, json=payload, timeout=60)
            response.raise_for_status()
            
            data = response.json()
            raw_content = data['choices'][0]['message']['content']
            
            # Parse JSON from response
            json_match = re.search(r'\{[\s\S]*\}', raw_content)
            if json_match:
                result = json.loads(json_match.group())
            else:
                result = json.loads(raw_content)
            
            return (result, raw_content, None)
            
        except requests.exceptions.ConnectionError:
            return (None, None, "connection_error")
        except json.JSONDecodeError as e:
            return (None, None, f"json_error: {e}")
        except Exception as e:
            return (None, None, str(e))
    
    def extract(self, query: str, temperature: float = 0.1, max_retries: int = 2, verbose: bool = False) -> Dict[str, Any]:
        """
        Extract slots from Indonesian e-commerce query with automatic retry.
        
        Args:
            query: Customer query in Indonesian
            temperature: Generation temperature (lower = more deterministic)
            max_retries: Number of retry attempts for invalid responses
            verbose: Print raw response
        
        Returns:
            Dictionary with task, entities, confidence, etc.
        """
        last_error = None
        
        for attempt in range(max_retries + 1):
            current_temp = min(0.5, temperature + (attempt * 0.1))  # Increase temp on retry
            
            result, raw_content, error = self._do_extract(query, current_temp)
            
            # Handle connection error - don't retry
            if error == "connection_error":
                return {
                    "task": "out_of_scope",
                    "entities": {},
                    "error": "Server not running. Start with: ./scripts/start_llm_server.sh"
                }
            
            if error:
                last_error = error
            
            if verbose and raw_content:
                print(f"Raw response (attempt {attempt + 1}): {raw_content}")
            
            # Check if response is valid
            if result and self._validate_response(result):
                return result
        
        # All retries failed
        return {
            "task": "out_of_scope",
            "entities": {},
            "multi_intent": [],
            "confidence": 0.0,
            "needs_clarification": True,
            "clarification_question": "Maaf, saya tidak memahami pertanyaan Anda. Bisa diulangi?",
            "error": f"extraction_failed_after_{max_retries + 1}_attempts: {last_error}"
        }


# Initialize extractor
extractor = SlotExtractor()
print("SlotExtractor initialized with Qwen3-4B-Instruct-2507!")
print(f"Server URL: {extractor.server_url}")
print(f"Max retries: 2")
print(f"Tasks supported: 10 (check_order, product_info, product_list, search_product, product_by_category, product_by_attribute, ask_price, check_stock, ask_payment, out_of_scope)")

SlotExtractor initialized with Qwen3-4B-Instruct-2507!
Server URL: http://localhost:8080
Max retries: 2
Tasks supported: 10 (check_order, product_info, product_list, search_product, product_by_category, product_by_attribute, ask_price, check_stock, ask_payment, out_of_scope)


## Step 5: Test Slot Extraction (All 10 Tasks)

Test queries covering all 10 task types:
- `check_order`: 3 queries (including one without order_id for clarification)
- `product_info`: 2 queries
- `product_list`: 2 queries
- `search_product`: 2 queries
- `product_by_category`: 2 queries
- `product_by_attribute`: 2 queries
- `ask_price`: 2 queries
- `check_stock`: 2 queries
- `ask_payment`: 2 queries
- `out_of_scope`: 2 queries
- Multi-intent: 2 queries

**Total: 23 test queries**

In [16]:
# Test queries covering all 10 tasks
test_queries = [
    # ============================================================
    # check_order - Status pesanan
    # ============================================================
    "pesanan saya 12345 udah sampai mana?",
    "psen gw 99871 gmn?",
    "order gue kapan nyampe ya",  # needs_clarification (no order_id)
    
    # ============================================================
    # product_info - Detail produk
    # ============================================================
    "bahan kaos ini apa ya",
    "spek laptop lenovo yang ini gimana?",
    
    # ============================================================
    # product_list - Daftar produk umum
    # ============================================================
    "produk ada apa aja?",
    "barang yang dijual apa saja?",
    
    # ============================================================
    # search_product - Cari produk via keyword
    # ============================================================
    "cari sepatu lari dong",
    "cariin tas ransel",
    
    # ============================================================
    # product_by_category - Produk berdasarkan kategori
    # ============================================================
    "produk fashion ada apa aja?",
    "kategori elektronik ada apa?",
    
    # ============================================================
    # product_by_attribute - Produk berdasarkan atribut
    # ============================================================
    "ada baju nike warna merah ukuran L?",
    "sepatu adidas size 42 warna hitam ada ga?",
    
    # ============================================================
    # ask_price - Harga produk
    # ============================================================
    "harga laptop asus berapa?",
    "brp hrg sepatu nike?",
    
    # ============================================================
    # check_stock - Stok produk
    # ============================================================
    "stok hp xiaomi masih ada?",
    "ready stock jaket biru?",
    
    # ============================================================
    # ask_payment - Metode pembayaran
    # ============================================================
    "bisa bayar pake gopay ga?",
    "cara bayar gimana sih",
    
    # ============================================================
    # out_of_scope - Di luar cakupan
    # ============================================================
    "mau refund dong barang rusak",
    "jam buka toko kapan ya",
    
    # ============================================================
    # Multi-intent examples
    # ============================================================
    "stok hp samsung ada ga? kalo ada harga berapa?",
    "jaket nike ready ga? gimana cara bayarnya?",
]

print("Testing slot extraction with 10 tasks (new system prompt)...")
print("=" * 70)
print(f"Total test queries: {len(test_queries)}")
print("=" * 70)

import time
results = []
latencies = []
task_counts = {}

for query in test_queries:
    print(f"\nQuery: \"{query}\"")
    
    start = time.time()
    result = extractor.extract(query)
    elapsed = time.time() - start
    latencies.append(elapsed)
    
    results.append({"query": query, "result": result, "latency": elapsed})
    
    if "error" in result and result.get("task") == "out_of_scope" and result.get("confidence") == 0.0:
        print(f"  ERROR: {result['error']}")
        break
    else:
        task = result.get('task', 'N/A')
        task_counts[task] = task_counts.get(task, 0) + 1
        
        print(f"  Task: {task}")
        
        # Show entities with new structure
        entities = result.get('entities', {})
        if entities:
            print(f"  Entities:")
            if entities.get('product_name'):
                print(f"    - product_name: {entities['product_name']}")
            if entities.get('category'):
                print(f"    - category: {entities['category']}")
            if entities.get('order_id'):
                print(f"    - order_id: {entities['order_id']}")
            if entities.get('quantity'):
                print(f"    - quantity: {entities['quantity']}")
            
            # Show attributes (new nested structure)
            attrs = entities.get('attributes', {})
            if attrs and any(attrs.values()):
                print(f"    - attributes:")
                if attrs.get('brand'):
                    print(f"        brand: {attrs['brand']}")
                if attrs.get('color'):
                    print(f"        color: {attrs['color']}")
                if attrs.get('size'):
                    print(f"        size: {attrs['size']}")
        
        if result.get('multi_intent'):
            print(f"  Multi-intent: {result.get('multi_intent')}")
        
        if result.get('needs_clarification'):
            print(f"  Needs clarification: {result.get('clarification_question')}")
        
        print(f"  Confidence: {result.get('confidence', 'N/A')}")
        print(f"  Latency: {elapsed:.2f}s")

print("\n" + "=" * 70)
print(f"Tested {len(results)} queries")
print("\nTask Distribution:")
for task, count in sorted(task_counts.items(), key=lambda x: -x[1]):
    print(f"  {task}: {count}")

Testing slot extraction with 10 tasks (new system prompt)...
Total test queries: 23

Query: "pesanan saya 12345 udah sampai mana?"
  Task: check_order
  Entities:
    - order_id: 12345
  Confidence: 0.95
  Latency: 9.28s

Query: "psen gw 99871 gmn?"
  Task: check_order
  Entities:
    - order_id: 99871
  Confidence: 0.95
  Latency: 3.34s

Query: "order gue kapan nyampe ya"
  Task: check_order
  Entities:
  Needs clarification: Bisakah Anda sebutkan nomor order Anda?
  Confidence: 0.85
  Latency: 3.10s

Query: "bahan kaos ini apa ya"
  Task: product_info
  Entities:
    - product_name: kaos
  Confidence: 0.9
  Latency: 2.81s

Query: "spek laptop lenovo yang ini gimana?"
  Task: product_info
  Entities:
    - product_name: laptop
    - attributes:
        brand: lenovo
  Confidence: 0.9
  Latency: 2.87s

Query: "produk ada apa aja?"
  Task: product_list
  Entities:
  Confidence: 0.85
  Latency: 2.78s

Query: "barang yang dijual apa saja?"
  Task: product_list
  Entities:
  Confidence: 0.

## Step 6: Evaluation Results

Comprehensive evaluation including:
- **Latency Statistics**: Mean, Std, Min, Max, Median, Throughput
- **Task Coverage**: Which of the 10 tasks were detected
- **Confidence Analysis**: Mean confidence, high confidence rate
- **Clarification Analysis**: Queries that needed clarification
- **Multi-intent Detection**: Queries with secondary intents
- **Entity Extraction**: Product names, categories, order IDs, attributes (brand, color, size)

In [17]:
import numpy as np

if latencies:
    print("=" * 70)
    print("EVALUATION RESULTS")
    print("=" * 70)
    
    print("\n📊 Latency Statistics (Native llama.cpp + Metal):")
    print(f"  Mean: {np.mean(latencies):.3f}s")
    print(f"  Std: {np.std(latencies):.3f}s")
    print(f"  Min: {np.min(latencies):.3f}s")
    print(f"  Max: {np.max(latencies):.3f}s")
    print(f"  Median: {np.median(latencies):.3f}s")
    print(f"\n  Throughput: ~{60/np.mean(latencies):.1f} queries/minute")
    
    print("\n📈 Task Coverage:")
    all_tasks = ['check_order', 'product_info', 'product_list', 'search_product', 
                 'product_by_category', 'product_by_attribute', 'ask_price', 
                 'check_stock', 'ask_payment', 'out_of_scope']
    covered = set(task_counts.keys())
    missing = set(all_tasks) - covered
    
    print(f"  Covered: {len(covered)}/{len(all_tasks)} tasks")
    if missing:
        print(f"  Missing: {missing}")
    else:
        print(f"  ✅ All 10 tasks covered!")
    
    print("\n📋 Confidence Analysis:")
    confidences = [r['result'].get('confidence', 0) for r in results if r['result'].get('confidence')]
    if confidences:
        print(f"  Mean confidence: {np.mean(confidences):.2f}")
        print(f"  Min confidence: {np.min(confidences):.2f}")
        print(f"  High confidence (>0.8): {sum(1 for c in confidences if c > 0.8)}/{len(confidences)}")
    
    # Check for needs_clarification
    clarification_needed = [r for r in results if r['result'].get('needs_clarification')]
    print(f"\n🔄 Needs Clarification: {len(clarification_needed)}/{len(results)}")
    for r in clarification_needed:
        print(f"  - \"{r['query'][:40]}...\" → {r['result'].get('clarification_question')}")
    
    # Check for multi-intent
    multi_intent_results = [r for r in results if r['result'].get('multi_intent')]
    print(f"\n🎯 Multi-intent Detected: {len(multi_intent_results)}/{len(results)}")
    for r in multi_intent_results:
        print(f"  - \"{r['query'][:40]}...\" → {r['result'].get('multi_intent')}")
    
    # Entity extraction accuracy
    print("\n📦 Entity Extraction Summary:")
    product_names = [r['result'].get('entities', {}).get('product_name') for r in results]
    categories = [r['result'].get('entities', {}).get('category') for r in results]
    order_ids = [r['result'].get('entities', {}).get('order_id') for r in results]
    
    print(f"  Product names extracted: {sum(1 for p in product_names if p)}")
    print(f"  Categories extracted: {sum(1 for c in categories if c)}")
    print(f"  Order IDs extracted: {sum(1 for o in order_ids if o)}")
    
    # Attribute extraction
    attrs_with_brand = sum(1 for r in results if r['result'].get('entities', {}).get('attributes', {}).get('brand'))
    attrs_with_color = sum(1 for r in results if r['result'].get('entities', {}).get('attributes', {}).get('color'))
    attrs_with_size = sum(1 for r in results if r['result'].get('entities', {}).get('attributes', {}).get('size'))
    
    print(f"  Attributes extracted:")
    print(f"    - brand: {attrs_with_brand}")
    print(f"    - color: {attrs_with_color}")
    print(f"    - size: {attrs_with_size}")
    
else:
    print("No latency data - server may not be running")

EVALUATION RESULTS

📊 Latency Statistics (Native llama.cpp + Metal):
  Mean: 3.191s
  Std: 1.306s
  Min: 2.758s
  Max: 9.278s
  Median: 2.873s

  Throughput: ~18.8 queries/minute

📈 Task Coverage:
  Covered: 10/10 tasks
  ✅ All 10 tasks covered!

📋 Confidence Analysis:
  Mean confidence: 0.90
  Min confidence: 0.70
  High confidence (>0.8): 22/23

🔄 Needs Clarification: 1/23
  - "order gue kapan nyampe ya..." → Bisakah Anda sebutkan nomor order Anda?

🎯 Multi-intent Detected: 2/23
  - "stok hp samsung ada ga? kalo ada harga b..." → ['ask_price']
  - "jaket nike ready ga? gimana cara bayarny..." → ['check_stock', 'ask_payment']

📦 Entity Extraction Summary:
  Product names extracted: 12
  Categories extracted: 2
  Order IDs extracted: 2
  Attributes extracted:
    - brand: 8
    - color: 3
    - size: 2


## Step 7: Interactive Testing

Test custom queries and validate the new entity structure:
- `product_name`: Product being referenced
- `category`: Product category
- `order_id`: Order number
- `quantity`: Requested quantity
- `attributes.brand`: Brand name (Nike, Adidas, etc.)
- `attributes.color`: Color (merah, biru, hitam, etc.)
- `attributes.size`: Size (S, M, L, XL, 42, etc.)

In [18]:
# Interactive testing - change this query to test different inputs
query = "ada baju nike warna merah ukuran L? harganya berapa?"

print(f"Query: \"{query}\"")
print("-" * 50)

start = time.time()
result = extractor.extract(query, verbose=True)
elapsed = time.time() - start

print(f"\nLatency: {elapsed:.2f}s")
print("\nParsed Result:")
print(json.dumps(result, indent=2, ensure_ascii=False))

# Validate entity structure
print("\n" + "-" * 50)
print("Entity Structure Validation:")
entities = result.get('entities', {})
print(f"  product_name: {entities.get('product_name')}")
print(f"  category: {entities.get('category')}")
print(f"  order_id: {entities.get('order_id')}")
print(f"  quantity: {entities.get('quantity')}")
attrs = entities.get('attributes', {})
print(f"  attributes:")
print(f"    brand: {attrs.get('brand') if attrs else None}")
print(f"    color: {attrs.get('color') if attrs else None}")
print(f"    size: {attrs.get('size') if attrs else None}")

Query: "ada baju nike warna merah ukuran L? harganya berapa?"
--------------------------------------------------
Raw response (attempt 1): {
  "task": "product_by_attribute",
  "entities": {
    "product_name": "baju",
    "category": null,
    "attributes": {
      "brand": "nike",
      "color": "merah",
      "size": "L"
    },
    "order_id": null,
    "quantity": null
  },
  "multi_intent": [
    "ask_price"
  ],
  "confidence": 0.95,
  "needs_clarification": false,
  "clarification_question": null
}

Latency: 3.31s

Parsed Result:
{
  "task": "product_by_attribute",
  "entities": {
    "product_name": "baju",
    "category": null,
    "attributes": {
      "brand": "nike",
      "color": "merah",
      "size": "L"
    },
    "order_id": null,
    "quantity": null
  },
  "multi_intent": [
    "ask_price"
  ],
  "confidence": 0.95,
  "needs_clarification": false,
  "clarification_question": null
}

--------------------------------------------------
Entity Structure Validation:
  pr

## Step 8: Save Results

In [19]:
from datetime import datetime
import platform

if results and not (results[0].get("result", {}).get("confidence") == 0.0 and "error" in results[0].get("result", {})):
    
    # Calculate accuracy metrics
    confidences = [r['result'].get('confidence', 0) for r in results if r['result'].get('confidence')]
    
    test_summary = {
        "timestamp": datetime.now().isoformat(),
        "model": "Qwen3-4B-Instruct-2507-GGUF (Q4_K_M)",
        "runtime": f"macOS {platform.mac_ver()[0]} - {platform.processor()}",
        "backend": "Native llama.cpp (Homebrew) + Metal GPU",
        "system_prompt_version": "v2.0 - 10 Tasks with Bahasa Indonesia",
        "total_queries": len(results),
        "tasks_supported": list(VALID_TASKS),
        "latency_stats": {
            "mean": float(np.mean(latencies)),
            "std": float(np.std(latencies)),
            "min": float(np.min(latencies)),
            "max": float(np.max(latencies)),
            "median": float(np.median(latencies)),
            "throughput_per_min": float(60/np.mean(latencies))
        },
        "accuracy_stats": {
            "mean_confidence": float(np.mean(confidences)) if confidences else 0,
            "high_confidence_rate": sum(1 for c in confidences if c > 0.8) / len(confidences) if confidences else 0,
            "needs_clarification_count": sum(1 for r in results if r['result'].get('needs_clarification')),
            "multi_intent_count": sum(1 for r in results if r['result'].get('multi_intent')),
        },
        "task_distribution": task_counts,
        "entity_extraction_stats": {
            "product_names": sum(1 for r in results if r['result'].get('entities', {}).get('product_name')),
            "categories": sum(1 for r in results if r['result'].get('entities', {}).get('category')),
            "order_ids": sum(1 for r in results if r['result'].get('entities', {}).get('order_id')),
            "attributes_brand": sum(1 for r in results if r['result'].get('entities', {}).get('attributes', {}).get('brand')),
            "attributes_color": sum(1 for r in results if r['result'].get('entities', {}).get('attributes', {}).get('color')),
            "attributes_size": sum(1 for r in results if r['result'].get('entities', {}).get('attributes', {}).get('size')),
        },
        "results": results
    }

    output_file = PROJECT_ROOT / "evaluation" / "llm_slot_extraction_10tasks_results.json"
    output_file.parent.mkdir(parents=True, exist_ok=True)

    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(test_summary, f, indent=2, ensure_ascii=False)

    print(f"✅ Results saved to: {output_file}")
    print(f"\nSummary:")
    print(f"  - Total queries: {len(results)}")
    print(f"  - Mean latency: {np.mean(latencies):.2f}s")
    print(f"  - Mean confidence: {np.mean(confidences):.2f}" if confidences else "  - Mean confidence: N/A")
    print(f"  - Tasks covered: {len(task_counts)}/10")
else:
    print("❌ No results to save - check if server is running")

✅ Results saved to: /Users/firas/Developer/skripsi-balor/evaluation/llm_slot_extraction_10tasks_results.json

Summary:
  - Total queries: 23
  - Mean latency: 3.19s
  - Mean confidence: 0.90
  - Tasks covered: 10/10


## Summary

### Setup (Native llama.cpp)

```bash
# 1. Install llama.cpp
brew install llama.cpp

# 2. Start server (in separate terminal)
./scripts/start_llm_server.sh

# 3. Run this notebook
```

### Configuration
- **Model**: Qwen3-4B-Instruct-2507-GGUF (Q4_K_M, 4-bit)
- **Size**: ~2.6 GB
- **Backend**: Native llama.cpp via Homebrew
- **Acceleration**: Metal GPU (Apple Silicon)
- **Context**: 4096 tokens
- **API**: OpenAI-compatible at http://localhost:8080

### System Prompt v2.0 - 10 Tasks

| Task | Description | Entity Focus |
|------|-------------|--------------|
| `check_order` | Status pesanan, tracking | order_id |
| `product_info` | Detail produk, spesifikasi | product_name |
| `product_list` | Daftar produk umum | - |
| `search_product` | Cari via keyword | product_name |
| `product_by_category` | Filter kategori | category |
| `product_by_attribute` | Filter atribut | attributes.brand/color/size |
| `ask_price` | Harga produk | product_name, attributes.brand |
| `check_stock` | Ketersediaan stok | product_name, attributes |
| `ask_payment` | Metode pembayaran | - |
| `out_of_scope` | Di luar cakupan | - |

### New Entity Structure

```json
{
  "entities": {
    "product_name": "string or null",
    "category": "string or null",
    "attributes": {
      "brand": "string or null",
      "color": "string or null",
      "size": "string or null"
    },
    "order_id": "string or null",
    "quantity": "number or null"
  }
}
```

### Expected Performance on M1 Pro
- **Latency**: ~1-3s per query
- **Throughput**: ~30-60 queries/minute
- **Memory**: ~3-4 GB
- **Confidence**: >0.85 average

### Features
- ✅ 10 task definitions (vs 6 in v1)
- ✅ Full Bahasa Indonesia support (formal & informal)
- ✅ Nested attributes structure (brand, color, size)
- ✅ Multi-intent detection
- ✅ Clarification requests when needed
- ✅ Slang/typo normalization (gw, ga, brp, hrg, dll)